# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset's Croissant metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We enumerate all record sets (`@id`), the fields of each record set, and the corresponding `@id` values.

In [ ]:
# Croissant uses `dataset.metadata.record_sets` for record set metadata

print("Available Record Sets:")
for record_set in metadata.record_sets:
    print(f"  - {record_set['@id']}  (name: {record_set['name']})")
    print("    Fields (@id, name, dataType):")
    for field in record_set.get('field', []):
        # Each field is a dict containing metadata
        f_id = field.get('@id', 'n/a')
        f_name = field.get('name', 'n/a')
        f_dtype = field.get('dataType', 'n/a')
        print(f"      - {f_id}, name: {f_name}, dataType: {f_dtype}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview above.

In this dataset, we use the main protein data table.

We'll list the record set ids and select the field ids for analysis.

In [ ]:
# List available record set @ids
record_sets = [rs['@id'] for rs in metadata.record_sets]
dataframes = {}

# Load all record sets into DataFrames
for rs_id in record_sets:
    print(f"Loading records for record set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"{rs_id}: loaded {len(df)} records, columns: {df.columns.tolist()}")
# Optionally show a sample DataFrame
main_rs_id = record_sets[0] if len(record_sets)>0 else None
if main_rs_id is not None:
    print(f"\nColumns of main record set ({main_rs_id}):")
    print(dataframes[main_rs_id].columns.tolist())
    display(dataframes[main_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes removing outliers, transforming data distributions, or grouping by attributes to prepare for analysis.

We'll demonstrate these steps with a numeric field such as molecular weight (`MW`) or coverage (`Coverage_percent`).

> **Note:** Please refer to section 2 for the exact `@id` for numeric fields in your dataset.

In [ ]:
# Please adjust field @ids as per section 2. Example field ids are used below.

main_rs_id = record_sets[0]  # Select the first record set (likely main table)
df = dataframes[main_rs_id]

# Choose a numeric field: get the field named 'MW' or similar from field ids
candidate_numeric_fields = [col for col in df.columns if ('MW' in col or 'Coverage' in col or 'Abundance' in col or 'PeptideCount' in col)]
if candidate_numeric_fields:
    numeric_field = candidate_numeric_fields[0]  # Use the first candidate
else:
    numeric_field = df.columns[0]  # Fallback

print(f"Using numeric field: {numeric_field}")

if pd.api.types.is_numeric_dtype(df[numeric_field]):
    threshold = df[numeric_field].quantile(0.9)  # Example: top 10% values
    filtered_df = df[df[numeric_field] > threshold].copy()
else:
    # Try to convert if possible
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
    threshold = df[numeric_field].quantile(0.9)
    filtered_df = df[df[numeric_field] > threshold].copy()

print(f"Filtered records with {numeric_field} > {threshold:.2f} (top 10%): {len(filtered_df)} rows")
display(filtered_df.head())

filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by a suitable categorical field (e.g., sample or protein type)
group_candidate_fields = [c for c in df.columns if c != numeric_field and df[c].dtype == 'O']
if group_candidate_fields:
    group_field = group_candidate_fields[0]
    print(f"Grouping by: {group_field}")
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram for the selected numeric field
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=30, kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If available, scatter plot two numeric features
if len(candidate_numeric_fields) > 1:
    plt.figure(figsize=(6,4))
    sns.scatterplot(data=df, x=candidate_numeric_fields[0], y=candidate_numeric_fields[1])
    plt.xlabel(candidate_numeric_fields[0])
    plt.ylabel(candidate_numeric_fields[1])
    plt.title(f"Scatter plot: {candidate_numeric_fields[0]} vs {candidate_numeric_fields[1]}")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we:
- Loaded FAIR^2 Croissant metadata and all available record sets.
- Identified key record set and field `@id` values for analysis.
- Performed filtering and normalization on a representative numeric field.
- Visualized data distributions.

Continue with dataset-specific analysis based on your experimental goals or project needs!